# K-Nearest Neighbors (KNN) Classification

This notebook explores KNN classification with the Wine dataset.

Goals:
- Understand why feature scaling matters for distance-based models.
- Build a reproducible preprocessing and modeling pipeline.
- Tune the number of neighbors with cross-validation.
- Evaluate the selected model on a held-out test set.

This is a supplementary, AI-assisted practice exercise. Run the cells, inspect the results, and adapt the experiments as part of your own learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 1. Load and split the data

The Wine dataset contains numeric measurements for three wine classes. We use a stratified split so class proportions are maintained in both subsets.

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", wine.target_names)

## 2. Build a pipeline

KNN relies on distances, so features with larger numeric scales can dominate. StandardScaler puts features on comparable scales. Keeping scaling inside a Pipeline also prevents data leakage during cross-validation.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 13],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best cross-validation accuracy: {search.best_score_:.3f}")

## 3. Evaluate on unseen data

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=wine.target_names,
    cmap="Blues"
)
plt.title("KNN — Confusion Matrix")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Experiment

Try changing the candidate values in `param_grid`. Compare the best parameters and test accuracy. Remember that repeatedly tuning against the test set makes it less useful as an unbiased final evaluation.